# EL F1 Score Matrices

Reads `scores.csv` and shows heatmaps with **models as rows**, **configs as columns**, and F1 scores as values — one heatmap per KB metric (Wikidata, GeoNames, NIL).

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

SCORES_CSV = Path("el-evaluation/scores.csv")

df = pd.read_csv(SCORES_CSV)

# Drop the bogus duplicate row (all metrics = 1.0)
df = df.drop_duplicates(subset=["source_text", "model", "temperature", "think"], keep="first")

# Create a readable config label
def make_config(row):
    parts = []
    if row["temperature"] == row["temperature"]:  # not NaN
        parts.append(f"temp={row['temperature']}")
    think = str(row.get("think", "")).strip().lower()
    if think and think not in ("", "nan"):
        parts.append(f"think={think}")
    return "  ".join(parts) if parts else "default"

df["config"] = df.apply(make_config, axis=1)

print(f"Loaded {len(df)} runs from {SCORES_CSV.name}")
print(f"Sources: {df['source_text'].unique()}")
print(f"Models:  {df['model'].unique()}")
print(f"Configs: {df['config'].unique()}")
df[["source_text", "model", "config", "wd_f1", "gn_f1", "nil_f1"]]

In [ ]:
# --- Run Tracker ---
# Scans el-results/ for completed _el.spacy files and shows a completion matrix.

EL_RESULTS = Path("el-results")
SOURCE = "1816_el_gs"

MODELS = ["deepseek-v4-flash", "gemma4:31b", "kimi-k2.7-code"]

CONFIGS = [
    ("false", 0.0),
    ("false", 0.5),
    ("false", 1.0),
    ("low", 1.0),
    ("medium", 1.0),
    ("high", 1.0),
]

rows = []
for model in MODELS:
    row = {"Model": model}
    for think, temp in CONFIGS:
        slug = model.replace(":", "-").replace("/", "-")
        think_part = f"think{think}" if think and think.lower() not in ("false", "") else "thinkfalse"
        filename = f"{SOURCE}__{slug}_t{temp}_{think_part}_el.spacy"
        col = f"think={think}  temp={temp}"
        row[col] = "✅" if (EL_RESULTS / filename).exists() else "❌"
    rows.append(row)

tracker = pd.DataFrame(rows).set_index("Model")
tracker["✅ Completed"] = (tracker.values == "✅").sum(axis=1)
tracker["❌ Remaining"] = (tracker.values == "❌").sum(axis=1)

print(f"Scanning: {EL_RESULTS.resolve()}")
print(f"Total combos: {len(MODELS)} models × {len(CONFIGS)} configs = {len(MODELS) * len(CONFIGS)}\n")
tracker

In [ ]:
def plot_f1_heatmap(data, metric_col, metric_label, source_filter=None, vmin=0, vmax=1):
    """Plot a heatmap with models as rows, configs as columns, F1 as values."""
    plot_df = data.copy()
    if source_filter:
        plot_df = plot_df[plot_df["source_text"] == source_filter].copy()

    pivot = plot_df.pivot_table(
        index="model", columns="config", values=metric_col, aggfunc="first"
    )

    fig, ax = plt.subplots(figsize=(max(2.5, len(pivot.columns) * 1.5), len(pivot.index) * 0.8 + 1))
    sns.heatmap(
        pivot, annot=True, fmt=".3f", cmap="YlGnBu",
        vmin=vmin, vmax=vmax, linewidths=0.5,
        cbar_kws={"label": f"{metric_label} F₁"},
        ax=ax,
    )
    ax.set_title(f"{metric_label} F₁  |  {source_filter or 'all sources'}", fontsize=13, fontweight="bold")
    ax.set_ylabel("")
    ax.set_xlabel("")
    plt.yticks(rotation=0)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    # Also print the raw numbers
    print(f"\n{metric_label} F₁  ({source_filter or 'all sources'}):")
    display(pivot.round(4))

## Wikidata F₁

In [ ]:
plot_f1_heatmap(df, "wd_f1", "Wikidata", source_filter="1816_el_gs")

## GeoNames F₁

In [ ]:
plot_f1_heatmap(df, "gn_f1", "GeoNames", source_filter="1816_el_gs")

## NIL F₁

In [ ]:
plot_f1_heatmap(df, "nil_f1", "NIL", source_filter="1816_el_gs")